In [1]:
# !pip install --upgrade wandb

In [2]:
# import os
# import wandb

# from kaggle_secrets import UserSecretsClient

# try:
#     user_secrets = UserSecretsClient()
#     wandb_key = user_secrets.get_secret("WANDB_API_KEY")
    
#     wandb.login(key=wandb_key)
#     print("✓ W&B authenticated via Kaggle secrets")
    
# except Exception as e:
#     print(f"⚠ W&B login failed: {e}")
#     print("Proceeding without W&B logging...")

In [3]:
from pathlib import Path

MESSY_DATA_DIR = Path("/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup")
GENRES_PT = Path("/kaggle/input/datasets/vishalladoiya/pre-compute-spec/genres_pt/genres_pt")
ESC_50_MASTER = MESSY_DATA_DIR / "ESC-50-master"
GENRES_STEMS = MESSY_DATA_DIR / "genres_stems"
MASHUPS = MESSY_DATA_DIR / "mashups"

json_path = "/kaggle/input/datasets/vishalladoiya/model-paths-2/modelpaths.json"

# Model Architecture  

In [4]:
import torch
import torch.nn as nn
from transformers import ASTForAudioClassification, ASTConfig


class ASTGenreClassifier(nn.Module):
    """
    ASTGenreClassifier

    A fine-tuned Audio Spectrogram Transformer (AST) for music genre classification.

    Backbone : MIT/ast-finetuned-audioset-10-10-0.4593
    Input    : Log-Mel spectrogram  [B, 1024, 128]  (or [B, 1, 1024, 128])
    Output   : Logits               [B, num_classes]

    Custom head:
        LayerNorm(768)
        Linear(768 -> 512) -> GELU -> Dropout(0.3)
        Linear(512 -> 256) -> GELU -> Dropout(0.2)
        Linear(256 -> num_classes)
    """

    def __init__(
            self, num_classes: int,
             time_frames: int = 1024,
             mel_bins: int = 128,
        ):
        """
        Args:
            num_classes : Number of output genre classes.
            time_frames : Spectrogram time frames (default 1024).
            mel_bins    : Mel frequency bins (default 128).
            dropout_1   : Dropout after first hidden layer (default 0.3).
            dropout_2   : Dropout after second hidden layer (default 0.2).
        """
        super().__init__()

        config = ASTConfig.from_pretrained(
            "MIT/ast-finetuned-audioset-10-10-0.4593"
        )
        config.num_labels = num_classes

        self.ast = ASTForAudioClassification.from_pretrained(
            "MIT/ast-finetuned-audioset-10-10-0.4593",
            config=config,
            ignore_mismatched_sizes=True,
        )

        self.time_frames = time_frames
        self.mel_bins = mel_bins
        self.num_classes = num_classes

        hidden_size = self.ast.config.hidden_size  # 768

        self.ast.classifier = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, 512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    @property
    def classifier(self):
        """
        Returns the classifier head of the AST model.
        """
        return self.ast.classifier

    def forward(self, input_values: torch.Tensor) -> torch.Tensor:
        """
        Forward pass of the model.

        Args:
            input_values (torch.Tensor):
                Log-Mel spectrogram tensor.
                Shape:
                    [batch_size, 1024, 128]
                or
                    [batch_size, 1, 1024, 128]

        Returns:
            torch.Tensor:
                Logits of shape [batch_size, num_classes]
        """

        # Remove channel dimension if present
        if input_values.dim() == 4:
            input_values = input_values.squeeze(1)

        outputs = self.ast(input_values=input_values)

        return outputs.logits
    


2026-03-16 18:34:20.201059: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773686060.378244      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773686060.430267      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773686060.850415      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773686060.850455      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773686060.850457      24 computation_placer.cc:177] computation placer alr

In [5]:
import torch
import torch.nn as nn
from transformers import Wav2Vec2Model


class Wav2Vec2ForGenre(nn.Module):

    """
    Wav2Vec2ForGenre

    Fine-tuned Wav2Vec2 model for music genre classification.

    Backbone : facebook/wav2vec2-base
    Input    : raw waveform [B, 163840]  (10.24s @ 16kHz)
               attention_mask [B, 163840]
    Output   : logits [B, num_classes]

    Architecture:
        Wav2Vec2 CNN feature extractor  (always frozen)
        Wav2Vec2 Transformer encoder    (unfrozen during fine-tuning)
        Mean pool over time             [B, T, 768] -> [B, 768]
        LayerNorm(768)
        Linear(768 -> 512) -> GELU -> Dropout(0.3)
        Linear(512 -> 256) -> GELU -> Dropout(0.2)
        Linear(256 -> num_classes)

    Notes:
        - CNN feature extractor is always frozen — it's a fixed filterbank,
          fine-tuning it rarely helps and wastes VRAM.
        - Attention-mask-weighted pooling is used instead of plain mean
          so padded positions don't dilute the representation.
        - Head uses GELU (smoother gradients) instead of ReLU.
        - LayerNorm before the head stabilises training, especially with
          the large LR used in Stage 1.
    """

    def __init__(self, num_classes: int, dropout_1: float = 0.3, dropout_2: float = 0.2):

        """
        Args:
            num_classes : Number of output genre classes.
            dropout_1   : Dropout after first hidden layer (default 0.3).
            dropout_2   : Dropout after second hidden layer (default 0.2).
        """

        super().__init__()

        self.wav2vec = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")

        self.wav2vec.feature_extractor._freeze_parameters()

        hidden_size  = self.wav2vec.config.hidden_size   # 768
        self.num_classes = num_classes

        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, 512),
            nn.GELU(),
            nn.Dropout(dropout_1),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(dropout_2),
            nn.Linear(256, num_classes),
        )

    def forward(self, input_values: torch.Tensor, attention_mask: torch.Tensor = None) -> torch.Tensor:
        """
        Args:
            input_values  : [B, 163840]  raw waveform
            attention_mask: [B, 163840]  1 = real sample, 0 = padding
                            Optional — if None, plain mean pooling is used.
        Returns:
            logits : [B, num_classes]
        """
        outputs = self.wav2vec(
            input_values=input_values,
            attention_mask=attention_mask,
        )

        hidden_states = outputs.last_hidden_state   # [B, T, 768]

        # Attention-mask-weighted pooling:
        # Wav2Vec2's CNN downsamples the mask from [B, 163840] to [B, T].
        if attention_mask is not None:
            # get_feat_extract_output_lengths maps waveform length -> T
            out_lengths = self.wav2vec._get_feat_extract_output_lengths(
                attention_mask.sum(dim=1)
            )                                        # [B]
            # Build a frame-level mask [B, T]
            B, T, _ = hidden_states.shape
            frame_mask = torch.arange(T, device=hidden_states.device).unsqueeze(0) < out_lengths.unsqueeze(1)  # [B, T]
            frame_mask = frame_mask.unsqueeze(-1).float()   # [B, T, 1]

            pooled = (hidden_states * frame_mask).sum(dim=1) / frame_mask.sum(dim=1).clamp(min=1e-9)  # [B, 768]
        else:
            pooled = hidden_states.mean(dim=1)       # [B, 768]

        return self.classifier(pooled)
        

In [6]:
import torch 
import torch.nn as nn 
import torch.nn.functional as F
from transformers import Wav2Vec2FeatureExtractor, AutoModel

class WeightedLayerAggregator(nn.Module):
    """
    Learnable weighted aggregation of all transformer layers.
    
    Aggregates features from all 24 transformer layers plus input embeddings
    using learned weights, which helps capture both low-level acoustic features
    (early layers) and high-level semantic information (late layers).
    """

    def __init__(self, num_layers: int = 25, hidden_size: int = 1024):
        """
        Args:
            num_layers  : Number of layers to aggregate (24 transformer + 1 input embedding).
            hidden_size : Hidden dimension size (1024 for MERT-330M).
        """
        super().__init__()

        self.num_layers = num_layers 
        self.layer_weights = nn.Parameter(torch.ones(num_layers) / num_layers)

    def forward(self, all_hidden_states):
        """
        Args:
            all_hidden_states : Tuple of tensors, each [batch, seq_len, hidden_size]
        
        Returns:
            torch.Tensor : Aggregated features [batch, seq_len, hidden_size]
        """

        # stack all layers: [num_layers, batch, seq_len, hidden_size]
        stacked = torch.stack(all_hidden_states, dim=0)
        weights = F.softmax(self.layer_weights, dim=0)

        # Weighted sum: [num_layers, 1, 1, 1] * [num_layers, batch, seq, hidden]
        weights = weights.view(-1, 1, 1, 1)
        aggregated = (stacked * weights).sum(dim=0)
        
        return aggregated
    


class MERTGenreClassifier(nn.Module):
    """
    MERTGenreClassifier
    
    A fine-tuned Music undERstanding model with large-scale self-supervised Training 
    (MERT) for music genre classification.
    
    Backbone : m-a-p/MERT-v1-330M
    Input    : Raw audio waveform [B, audio_length] at 24kHz
               Recommended audio_length: 30 seconds = 720,000 samples
    Output   : Logits [B, num_classes]
    
    Architecture:
        - MERT-v1-330M backbone (330M params, 24 transformer layers)
        - Weighted layer aggregation (learnable combination of all 24 layers)
        - Temporal average pooling
        - Custom classification head:
            LayerNorm(1024)
            Linear(1024 -> 512) -> ReLU -> Dropout(0.3)
            Linear(512 -> 256) -> ReLU -> Dropout(0.3)
            Linear(256 -> num_classes)
    """

    def __init__(
        self,
        num_classes: int,
        model_name: str = "m-a-p/MERT-v1-330M",
        sample_rate: int = 24000,
        chunk_duration: int = 30,
        use_weighted_layers: bool = True,
    ):
        
        """
        Args:
            num_classes         : Number of output genre classes.
            model_name          : HuggingFace model identifier (default MERT-v1-330M).
            sample_rate         : Audio sample rate in Hz (must be 24000 for MERT).
            chunk_duration      : Expected audio chunk duration in seconds (default 30).
            use_weighted_layers : Whether to use weighted layer aggregation (recommended).
        """

        super().__init__()
        
        self.num_classes = num_classes
        self.model_name = model_name
        self.sample_rate = sample_rate
        self.chunk_duration = chunk_duration
        self.use_weighted_layers = use_weighted_layers

        self.expected_length = int(chunk_duration * sample_rate) 

        # Load MERT backbone
        self.mert = AutoModel.from_pretrained(
            model_name,
            trust_remote_code=True
        )


        # Get model dimensions
        self.hidden_size = self.mert.config.hidden_size  # 1024 for MERT-330M
        self.num_layers = self.mert.config.num_hidden_layers 


        # Weighted layer aggregation
        if use_weighted_layers:
            self.layer_aggregator = WeightedLayerAggregator(
                num_layers=self.num_layers + 1,  # +1 for input embeddings
                hidden_size=self.hidden_size
            )
        else:
            self.layer_aggregator = None

        # Classification head
        self.classifier = nn.Sequential(
            nn.LayerNorm(self.hidden_size),
            nn.Linear(self.hidden_size, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

        # Feature extractor (processor)
        self.processor = Wav2Vec2FeatureExtractor.from_pretrained(
            model_name,
            trust_remote_code=True
        )

    
    def forward(self, input_values: torch.Tensor) -> torch.Tensor:
        """
        Forward pass of the model.
        
        Args:
            input_values (torch.Tensor):
                Raw audio waveform tensor.
                Shape: 
                    [batch_size, audio_length]
                where audio_length = chunk_duration * sample_rate
                Example: [8, 720000] for 30-second chunks at 24kHz
        
        Returns:
            torch.Tensor:
                Logits of shape [batch_size, num_classes]
        """
        # Get MERT features with all hidden states
        outputs = self.mert(
            input_values,
            output_hidden_states=True,
            return_dict=True
        )

        # Aggregate layers or use last layer
        if self.use_weighted_layers and self.layer_aggregator is not None:
            # Weighted combination of all 25 layers (input + 24 transformer layers)
            hidden_states = self.layer_aggregator(outputs.hidden_states)
        else:
            # Use only the last transformer layer
            hidden_states = outputs.last_hidden_state
        
        # hidden_states shape: [batch, seq_len, hidden_size]
        # seq_len ≈ 75 embeddings/second * 30 seconds = 2250 time steps
        
        # Temporal average pooling over time dimension
        pooled = hidden_states.mean(dim=1)  # [batch, hidden_size]
        
        # Classification
        logits = self.classifier(pooled)  # [batch, num_classes]
        
        return logits
    
    
    def get_features(self, input_values: torch.Tensor) -> torch.Tensor:
        """
        Extract MERT features without classification.
        Useful for feature extraction or transfer learning.
        
        Args:
            input_values : Raw audio [batch, audio_length]
        
        Returns:
            torch.Tensor : Pooled features [batch, hidden_size]
        """
        outputs = self.mert(
            input_values,
            output_hidden_states=True,
            return_dict=True
        )
        
        if self.use_weighted_layers and self.layer_aggregator is not None:
            hidden_states = self.layer_aggregator(outputs.hidden_states)
        else:
            hidden_states = outputs.last_hidden_state
        
        pooled = hidden_states.mean(dim=1)
        
        return pooled
    
    
    def count_parameters(self) -> dict:
        """
        Count trainable and total parameters.
        
        Returns:
            dict : Dictionary with parameter counts
        """
        total = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        
        return {
            'total': total,
            'trainable': trainable,
            'frozen': total - trainable,
            'trainable_pct': 100 * trainable / total if total > 0 else 0
        }

# prediction or inference code

Predict the music genre of an audio file.

      - All chunks are processed in batches (faster, avoids repeated overhead)
      - Combines BOTH soft-vote (avg probabilities) and hard-vote (majority)
        and uses soft-vote as the primary result (more accurate)
      - Returns both voting results so callers can inspect agreement
      - Graceful handling of very short files (< 1 chunk)

In [7]:
import os 
import json
import copy
import math
import pandas as pd
import numpy as np

import torch 
import torch.nn as nn
import torchaudio

from collections import Counter
from typing import Callable, Dict, List, Optional
from tqdm import tqdm
from pathlib import Path

from transformers import ASTFeatureExtractor, Wav2Vec2FeatureExtractor
from scipy.signal import butter, filtfilt


def highpass_filter(
    audio: torch.Tensor, 
    cutoff: int = 40, 
    sample_rate: int = 16000
):

    """
    Applies a Butterworth high-pass filter to remove very low-frequency noise.

    Args:
        audio (torch.Tensor): Input waveform tensor of shape [1, samples].
        cutoff (int): Cutoff frequency in Hz. Default is 40 Hz.
        sample_rate (int): Sampling rate of the waveform. Default is 16000 Hz.

    Returns:
        torch.Tensor: Filtered waveform with same shape and dtype as input.
    """

    b, a = butter(5, cutoff / (sample_rate / 2), btype='high')

    audio_np = audio.squeeze(0).cpu().numpy()

    filtered = filtfilt(b, a, audio_np)
    filtered = np.ascontiguousarray(filtered)
    filtered = torch.from_numpy(filtered).unsqueeze(0)

    return filtered.type_as(audio)



def load_ensemble_(model_fn: Callable, model_paths: List[str], device: str) -> List[nn.Module]:
    """
    Load multiple saved fold models into a list, ready for ensemble inference.

    Args:
        model_fn    : Zero-argument callable returning a fresh model architecture.
        model_paths : List of .pth file paths (one per fold).
        device      : 'cuda' or 'cpu'.

    Returns:
        models : list[nn.Module]  (all in eval mode, on device)
    """

    models = []
    for path in model_paths: 
        # Create a deep copy so each fold gets its own instance in memory
        m: nn.Module = copy.deepcopy(model_fn()).to(device) 
        m.load_state_dict(torch.load(path, map_location=device))
        m.eval()
        models.append(m)
        print(f" loaded {path}")

    return models



def load_ensemble(
    ast_model_fn: Callable, 
    mert_model_fn: Callable, 
    json_path: str, 
    device: str
) -> Dict[str, List[nn.Module]]:
 
    """
    Auto-discovers fold model paths from the provided JSON and loads them into an ensemble dictionary.

    Returns:
        models: Dictionary containing lists of initialized models for both AST and Wav2Vec2.
    """

    with open(json_path) as f:
        paths = json.load(f)

    model_paths_dict = paths["model_paths"]

    ast_paths = list(model_paths_dict["ast-model"].values())
    mert_paths = list(model_paths_dict["mert-model"].values())

    print(f"Loading {len(ast_paths)} AST models and {len(mert_paths)} Wav2Vec2 models from {json_path} ...")
    

    ast_models = load_ensemble_(
        model_fn=ast_model_fn, 
        model_paths=ast_paths, 
        device=device
    )

    mert_models = load_ensemble_(
        model_fn=mert_model_fn, 
        model_paths=mert_paths, 
        device=device
    )

    return {
        "ast-models": ast_models,
        "mert-models": mert_models
    }


class PredictionPipeline:
    """
    End-to-end inference pipeline for AST + MERT ensemble genre classification.

    Supports:
        - Multi-model ensemble prediction for both AST and MERT
        - Song-level prediction via chunk aggregation
        - Soft voting: weighted average of AST and MERT probabilities
        - Hard voting: majority vote over per-chunk argmaxes (diagnostic)
        - Temperature scaling for calibration
        - Batch or single-sample processing

    Args:
        models: {"ast-models": [...], "mert-models": [...]}
        genre_map: {genre_name: label_id}
        batch_size: chunks per forward pass
        ratio: weight of MERT in final blend (0 = AST only, 1 = MERT only)
               None = auto-weight by model count
        temperature: softmax temperature for probability calibration

    Usage:
        models = load_ensemble(ast_fn, mert_fn, "paths.json", device)
        pipeline = HybridPredictionPipeline(models, genre_map)
        genre, confidence, probs = pipeline.predict("song.mp3", verbose=True)
    """

    AST_SAMPLE_RATE: int = 16000
    AST_CHUNK_SIZE = 10.24
    AST_CHUNK_SAMPLES = int(AST_SAMPLE_RATE * AST_CHUNK_SIZE)

    MERT_SAMPLE_RATE = 24000
    MERT_CHUNK_SIZE = 30
    MERT_CHUNK_SAMPLES = int(MERT_SAMPLE_RATE * MERT_CHUNK_SIZE)

    def __init__(
        self, 
        models: Dict[str, List[nn.Module]],
        genre_map: dict,
        batch_size: int = 32,
        ratio: float = None,
        temperature: float = 1.0,
    ):
        
        self.models = models
        self.genre_map = genre_map
        self.index_to_genre = {v: k for k, v in self.genre_map.items()}
        self.batch_size = batch_size 
        self.temperature = temperature
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        n_ast = len(self.models["ast-models"])
        n_mert = len(self.models["mert-models"])
        if ratio is None:
            self.ast_weight = n_ast / (n_ast + n_mert)
            self.mert_weight = n_mert / (n_ast + n_mert)
        else:
            self.ast_weight = 1 - ratio
            self.mert_weight = ratio

        self.ast_processor = ASTFeatureExtractor.from_pretrained(
            "MIT/ast-finetuned-audioset-10-10-0.4593"
        )
        self.mert_processor = Wav2Vec2FeatureExtractor.from_pretrained(
            "m-a-p/MERT-v1-330M", 
            trust_remote_code=True
        )

        for key in self.models.keys():
            for m in self.models[key]:
                m.to(self.device)
                m.eval()


    def _load_and_preprocess_ast(self, audio_path: Path) -> torch.Tensor:
        """Load audio for AST: 16kHz, highpass filtered."""

        waveform, sr = torchaudio.load(audio_path)
        waveform = waveform.mean(dim=0, keepdim=True)

        if sr != self.AST_SAMPLE_RATE:
            resampler = torchaudio.transforms.Resample(sr, self.AST_SAMPLE_RATE)
            waveform = resampler(waveform)

        waveform = highpass_filter(
            audio=waveform, 
            cutoff=40, 
            sample_rate=self.AST_SAMPLE_RATE
        )

        return waveform
    

    def _load_and_preprocess_mert(self, audio_path: Path) -> torch.Tensor:
        """Load audio for MERT: 24kHz, highpass filtered."""

        waveform, sr = torchaudio.load(audio_path)
        waveform = waveform.mean(dim=0, keepdim=True)

        if sr != self.MERT_SAMPLE_RATE:
            resampler = torchaudio.transforms.Resample(sr, self.MERT_SAMPLE_RATE)
            waveform = resampler(waveform)

        waveform = highpass_filter(
            audio=waveform, 
            cutoff=40, 
            sample_rate=self.MERT_SAMPLE_RATE
        )
        return waveform

    
    def _chunk_wavefrom_ast(self, waveform: torch.Tensor) -> List[torch.Tensor]:
        """Split waveform into fixed-length chunks; pad the last one if needed."""

        total_samples = int(waveform.shape[1])
        n_chunks = max(1, math.ceil(total_samples / self.AST_CHUNK_SAMPLES))

        chunks = []
        for c in range(n_chunks):
            start = c * self.AST_CHUNK_SAMPLES 
            end = start + self.AST_CHUNK_SAMPLES 

            if end > total_samples:
                chunk = waveform[:, start:]
                padding = self.AST_CHUNK_SAMPLES - chunk.shape[1]

                if padding > 0:
                    chunk = nn.functional.pad(chunk, (0, padding))

            else:
                chunk = waveform[:, start:end]

            chunks.append(chunk)

        return chunks
    

    def _chunk_waveform_mert(self, waveform: torch.Tensor) -> List[torch.Tensor]:
        """Split waveform into 30s chunks for MERT."""
        total_samples = int(waveform.shape[1])
        
        # If song is shorter than 30s, pad it and return 1 chunk
        if total_samples < self.MERT_CHUNK_SAMPLES:
            chunk = waveform[:, :]
            padding = self.MERT_CHUNK_SAMPLES - total_samples
            chunk = torch.nn.functional.pad(chunk, (0, padding))
            return [chunk]
        
        n_chunks = total_samples // self.MERT_CHUNK_SAMPLES        
        chunks = []

        for c in range(n_chunks):
            start = c * self.MERT_CHUNK_SAMPLES
            end = start + self.MERT_CHUNK_SAMPLES

            if end > total_samples:
                chunk = waveform[:, start:]
                padding = self.MERT_CHUNK_SAMPLES - chunk.shape[1]
                if padding > 0:
                    chunk = torch.nn.functional.pad(chunk, (0, padding))
            else:
                chunk = waveform[:, start:end]

            chunks.append(chunk)

        return chunks
    

    @staticmethod
    def _normalize_chunk(chunk: torch.Tensor) -> torch.Tensor:
        """RMS normalization — same as training pipeline."""
        rms = torch.sqrt(torch.mean(chunk ** 2) + 1e-9)
        chunk = chunk / (rms * 3.0)
        return chunk.clamp(-1.0, 1.0)
    

    def _chunks_to_spectrograms(self, chunks: List[torch.Tensor]) -> torch.Tensor:
        """
        Convert a list of 1-D waveform chunks to a batched spectrogram tensor.
        Returns shape: [N, time_steps, freq_bins]
        Processing the whole song in one feature_extractor call is faster and
        ensures identical padding behaviour across all chunks.
        """

        numpy_chunks = [c.squeeze(0).cpu().numpy() for c in chunks]

        inputs = self.ast_processor(
            numpy_chunks,
            sampling_rate=self.AST_SAMPLE_RATE,
            return_tensors="pt"
        )

        return inputs.input_values.to(self.device)   # [N, 1024, 128]
    

    def _chunks_to_mert_input(self, chunks: List[torch.Tensor]) -> torch.Tensor:
        """Process waveform chunks for MERT."""
        # Normalize chunks
        normalized = []

        for chunk in chunks:
            rms = torch.sqrt(torch.mean(chunk ** 2) + 1e-9)
            chunk = chunk / (rms * 3.0)
            chunk = chunk.clamp(-1.0, 1.0)
            normalized.append(chunk.squeeze(0).cpu().numpy())

        # Process with MERT processor
        inputs = self.mert_processor(
            normalized,
            sampling_rate=self.MERT_SAMPLE_RATE,
            padding=True,
            return_tensors="pt"
        )

        return inputs.input_values

    
    @torch.no_grad()
    def _get_chunk_probs_for_ast_for_batch(self, spectrograms: torch.Tensor) -> torch.Tensor:
        """
        Run all chunks through all models in batches.

        Returns:
            chunk_probs : [N, num_classes]  — averaged over ensemble models.
        """

        n_chunks = len(spectrograms)
        models_chunk_probs = []

        for model in self.models["ast-models"]:
            model_probs = []

            for i in range(0, n_chunks, self.batch_size):
                batch = spectrograms[i: i+self.batch_size].to(self.device)
                logits = model(batch)

                probs = torch.softmax(logits, dim=1).cpu()
                model_probs.append(probs)

            models_chunk_probs.append(torch.cat(model_probs, dim=0))

        # Average across ensemble models → [N, C]
        ensemble_probs = torch.stack(models_chunk_probs).mean(dim=0)
        return ensemble_probs
    

    @torch.no_grad()
    def _get_mert_chunk_probs_for_batch(self, waveforms: torch.Tensor) -> torch.Tensor:
        """
        Get averaged predictions from MERT ensemble for all chunks.
        
        Args:
            waveforms: [N, 720000]
        
        Returns:
            chunk_probs: [N, num_classes]
        """
        n_chunks = len(waveforms)
        all_model_probs = []

        for model in self.models["mert-models"]:
            model_probs = []

            for i in range(0, n_chunks, self.batch_size):
                batch = waveforms[i:i + self.batch_size].to(self.device)
                logits = model(batch)
                probs = torch.softmax(logits, dim=1).cpu()
                model_probs.append(probs)

            all_model_probs.append(torch.cat(model_probs, dim=0))

        # Average across all MERT models
        ensemble_probs = torch.stack(all_model_probs).mean(dim=0)
        return ensemble_probs
    

    @torch.no_grad()
    def _get_chunk_probs_for_ast_for_single(self, spectrograms: torch.Tensor) -> torch.Tensor:
        """
        Run N individual spectrogram chunks through all AST fold models one at a time.
        spectrograms : [N, 1024, 128]
        
        Returns: 
            [N, C] averaged over ensemble.
        """

        n_chunks = spectrograms.shape[0]
        models_chunks_probs = []

        for model in self.models["ast-models"]:
            model_probs = []

            for i in range(0, n_chunks):
                chunk  = spectrograms[i].unsqueeze(0).to(self.device)
                logits = model(chunk)

                probs = torch.softmax(logits, dim=1).cpu()
                model_probs.append(probs)

            models_chunks_probs.append(torch.cat(model_probs, dim=0))

        ensemble_probs = torch.stack(models_chunks_probs).mean(dim=0)
        return ensemble_probs
    
    
    @torch.no_grad()
    def _get_mert_chunk_probs_single(self, waveforms: torch.Tensor) -> torch.Tensor:
        """
        Run N individual waveform chunks through all MERT models one at a time.
        Slower but uses less memory than batch processing.
        
        Args:
            waveforms: [N, 720000]
        
        Returns:
            chunk_probs: [N, num_classes] averaged over ensemble
        """
        
        n_chunks = len(waveforms)
        all_model_probs = []

        for model in self.models["mert-models"]:
            model_probs = []

            for i in range(n_chunks):
                chunk = waveforms[i].unsqueeze(0).to(self.device)
                logits = model(chunk)
                probs = torch.softmax(logits, dim=1).cpu()
                model_probs.append(probs)

            all_model_probs.append(torch.cat(model_probs, dim=0))

        # Average across all MERT models → [N, C]
        ensemble_probs = torch.stack(all_model_probs).mean(dim=0)
        return ensemble_probs


    def _apply_temperature(self, probs: torch.Tensor) -> torch.Tensor:
        """
        Soften overconfident probability distributions.
        T > 1 flattens distribution, T < 1 sharpens it.
        Applied before blending so neither model dominates unfairly.
        """
        if self.temperature == 1.0:
            return probs  # no-op, skip computation
        log_probs = torch.log(probs.clamp(min=1e-9)) / self.temperature
        return torch.softmax(log_probs, dim=-1)
    

    def _blend_and_vote(
        self,
        ast_probs: torch.Tensor,
        mert_probs: torch.Tensor,
        n_ast_chunks: int,
        n_mert_chunks: int,
        verbose: bool = False,
        audio_path: Path = None,
        return_json: bool = False,
    ) -> tuple | dict:
        """
        Blend AST and MERT probabilities and perform voting.
        
        Args:
            ast_probs: [N_ast, num_classes] probabilities from AST ensemble
            mert_probs: [N_mert, num_classes] probabilities from MERT ensemble
            n_ast_chunks: Number of AST chunks processed
            n_mert_chunks: Number of MERT chunks processed
            verbose: Print detailed prediction info
            audio_path: Path to audio file (for display)
            return_json: Return result as dict instead of tuple
        
        Returns:
            If return_json=False: (predicted_genre, confidence, all_probs_sorted)
            If return_json=True: dict with detailed prediction info
        """
        # Apply temperature scaling
        ast_probs = self._apply_temperature(ast_probs)
        mert_probs = self._apply_temperature(mert_probs)

        # Average across chunks for each model type
        ast_avg = ast_probs.mean(dim=0)   # [C]
        mert_avg = mert_probs.mean(dim=0)  # [C]

        # Weighted blend
        blended_avg = self.ast_weight * ast_avg + self.mert_weight * mert_avg  # [C]

        # Soft vote (primary prediction)
        soft_idx = blended_avg.argmax().item()
        soft_confidence = blended_avg[soft_idx].item()
        predicted_genre = self.index_to_genre[soft_idx]

        # Hard vote (diagnostic - per-chunk majority)
        # Use minimum number of chunks or interpolate
        min_chunks = min(len(ast_probs), len(mert_probs))
        
        # Take first min_chunks from each
        ast_probs_aligned = ast_probs[:min_chunks]
        mert_probs_aligned = mert_probs[:min_chunks]
        
        # Blend per-chunk probabilities
        blended_chunks = (self.ast_weight * ast_probs_aligned + self.mert_weight * mert_probs_aligned)  # [min_chunks, C]
        
        chunk_preds = blended_chunks.argmax(dim=1).tolist()
        hard_idx = Counter(chunk_preds).most_common(1)[0][0]
        hard_votes = chunk_preds.count(hard_idx)
        hard_genre = self.index_to_genre[hard_idx]

        # Sort all probabilities
        all_probs_sorted = dict(sorted(
            {self.index_to_genre[i]: blended_avg[i].item() for i in range(len(blended_avg))}.items(),
            key=lambda x: x[1], 
            reverse=True,
        ))

        if verbose:
            agreement = (
                "✅ agree" if soft_idx == hard_idx
                else f"⚠️ disagree (hard → {hard_genre} {hard_votes}/{len(chunk_preds)} chunks)"
            )
            print(f"\n[HybridPredictionPipeline]  {Path(audio_path).name if audio_path else 'audio'}")
            print(f"  AST chunks   : {n_ast_chunks}")
            print(f"  MERT chunks  : {n_mert_chunks}")
            print(f"  AST weight   : {self.ast_weight:.0%}  |  MERT weight: {self.mert_weight:.0%}")
            print(f"  Temperature  : {self.temperature}")
            print(f"  Soft vote    : {predicted_genre}  ({soft_confidence:.1%})")
            print(f"  Hard vote    : {hard_genre}  ({hard_votes}/{len(chunk_preds)} chunks)  {agreement}")
            print(f"  Top-3:")
            for genre, prob in list(all_probs_sorted.items())[:3]:
                print(f"      {genre:20s}: {prob:.1%}")

        if return_json:
            return {
                "genre": predicted_genre,
                "confidence": round(soft_confidence, 4),
                "all_probs": {k: round(v, 4) for k, v in all_probs_sorted.items()},
                "hard_vote": {
                    "genre": hard_genre,
                    "votes": hard_votes,
                    "chunks": len(chunk_preds),
                    "agree": soft_idx == hard_idx,
                },
                "weights": {
                    "ast": round(self.ast_weight, 3),
                    "mert": round(self.mert_weight, 3),
                },
                "chunks_info": {
                    "ast_chunks": n_ast_chunks,
                    "mert_chunks": n_mert_chunks,
                }
            }

        return predicted_genre, soft_confidence, all_probs_sorted


    def predict_batch(
        self, 
        audio_path: Path, 
        verbose: bool = False,
        return_json: bool = False
    ) -> tuple | dict:
        """
        Full song-level prediction using batch processing (faster).

        Blends AST and MERT chunk probabilities with soft voting:
            final = ast_weight * ast_avg + mert_weight * mert_avg

        Args:
            audio_path: Path to audio file
            verbose: Print detailed prediction info
            return_json: Return result as dict instead of tuple

        Returns:
            If return_json=False: (predicted_genre, confidence, all_probs_sorted)
            If return_json=True: dict with detailed prediction info
        """
        audio_path = Path(audio_path)

        # AST pipeline
        ast_waveform = self._load_and_preprocess_ast(audio_path)
        ast_chunks = self._chunk_wavefrom_ast(ast_waveform)
        ast_spectrograms = self._chunks_to_spectrograms(ast_chunks)
        
        # MERT pipeline
        mert_waveform = self._load_and_preprocess_mert(audio_path)
        mert_chunks = self._chunk_waveform_mert(mert_waveform)
        mert_inputs = self._chunks_to_mert_input(mert_chunks)

        # Batch processing
        ast_probs = self._get_chunk_probs_for_ast_for_batch(ast_spectrograms)
        mert_probs = self._get_mert_chunk_probs_for_batch(mert_inputs)

        # Blend and vote
        return self._blend_and_vote(
            ast_probs=ast_probs,
            mert_probs=mert_probs,
            n_ast_chunks=len(ast_chunks),
            n_mert_chunks=len(mert_chunks),
            verbose=verbose,
            audio_path=audio_path,
            return_json=return_json
        )
    

    def single_prediction(self, audio_path: Path, verbose: bool = False, return_json: bool = False) -> tuple | dict:

        """
        Full song-level prediction for single song.

        Blends AST and Wav2Vec2 chunk probabilities with soft voting:
            final = (1 - ratio) * ast_avg + ratio * wav2vec2_avg

        Returns:
            predicted_genre : str
            confidence      : float   (winning class probability)
            all_probs       : dict    {genre: prob} sorted descending
        """

        waveform = self._load_and_preprocess_ast(audio_path=audio_path)
        ast_chunks = self._chunk_wavefrom_ast(waveform=waveform)
        ast_input = self._chunks_to_spectrograms(chunks=ast_chunks)

        mert_waveform = self._load_and_preprocess_mert(audio_path)
        mert_chunks = self._chunk_waveform_mert(mert_waveform)
        mert_inputs = self._chunks_to_mert_input(mert_chunks)

        ast_probs = self._get_chunk_probs_for_ast_for_single(ast_input)
        mert_probs = self._get_mert_chunk_probs_single(mert_inputs)

        return self._blend_and_vote(
            ast_probs=ast_probs,
            mert_probs=mert_probs,
            n_ast_chunks=len(ast_chunks),
            n_mert_chunks=len(mert_chunks),
            verbose=verbose,
            audio_path=audio_path,
            return_json=return_json
        )
        

    def predict_dataset(self, test_df: pd.DataFrame, data_dir: Path, mode:str="batch") -> pd.DataFrame:
        """
        Run inference on every row in test_df.

        Args:
            test_df  : DataFrame with columns ["id", "filename"]
            data_dir : Root directory containing the audio files

        Returns:
            DataFrame with columns ["id", "genre"]
        """

        assert mode in ("batch", "single"), "mode must be 'batch' or 'single'"
        
        predict_fn = self.predict_batch if mode == "batch" else self.single_prediction

        print(f"\nStarting inference on {len(test_df)} files")
        print(f"  AST models:   {len(self.models['ast-models'])}")
        print(f"  MERT models:  {len(self.models['mert-models'])}")
        print(f"  Mode:         {mode}")
        print()


        results = {}

        for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Classifying"):
            song_id = row["id"]
            audio_path = data_dir / row["filename"]

            try :
                predicted_genre, _, _ = predict_fn(
                    audio_path=audio_path,
                    verbose=False,
                    return_json=False
                )
                results[song_id] = predicted_genre
            except Exception as e:
                print(f"\n  Error on {row['filename']} (id={song_id}): {e}")
                results[song_id] = "blues"

        
        results_df = pd.DataFrame(
            list(results.items()), columns=["id", "genre"]
        ).set_index("id")

        return results_df


In [8]:
import torch 
import pandas as pd
from pathlib import Path

device = "cuda" if torch.cuda.is_available() else "cpu"

num_classes = 10

genre_map = {
    'blues': 0, 'classical': 1, 'country': 2, 'disco': 3, 'hiphop': 4,
    'jazz': 5, 'metal': 6, 'pop': 7, 'reggae': 8, 'rock': 9
}

ast_model_fn = lambda: ASTGenreClassifier(num_classes=num_classes)
mert_model_fn = lambda: MERTGenreClassifier(num_classes=num_classes)

models = load_ensemble(
    ast_model_fn = ast_model_fn,
    mert_model_fn = mert_model_fn, 
    json_path = json_path,
    device = device
)

pipeline = PredictionPipeline(
    models = models,
    genre_map = genre_map, 
    batch_size = 16, 
    ratio = 0.5,
    temperature = 1.0 
)


test_df = pd.read_csv(MESSY_DATA_DIR / "test.csv") 

result = pipeline.predict_dataset(
    test_df = test_df,
    data_dir = MESSY_DATA_DIR,
    
)
result.to_csv("output.csv")

Loading 8 AST models and 4 Wav2Vec2 models from /kaggle/input/datasets/vishalladoiya/model-paths-2/modelpaths.json ...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
Y

 loaded /kaggle/input/models/vishalladoiya/ast-mert-models/pytorch/default/1/weights/ast-models/best_ast_genre_1.pth


Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


... (16 outputs left out)

 loaded /kaggle/input/models/vishalladoiya/ast-mert-models/pytorch/default/1/weights/mert-models/best_mert_genre_1.pth
 loaded /kaggle/input/models/vishalladoiya/ast-mert-models/pytorch/default/1/weights/mert-models/best_mert_genre_2.pth
 loaded /kaggle/input/models/vishalladoiya/ast-mert-models/pytorch/default/1/weights/mert-models/best_mert_genre_3.pth
 loaded /kaggle/input/models/vishalladoiya/ast-mert-models/pytorch/default/1/weights/mert-models/best_mert_genre_4.pth


preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]


Starting inference on 3020 files
  AST models:   8
  MERT models:  4
  Mode:         batch



Classifying:   0%|          | 0/3020 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch